# 02 · GP Surface and SVI Calibration

## Economic rationale

The cleaned surface from Notebook 01 is a sparse, noisy set of `(k, T) → IV`
points. We need two models that interpolate and extrapolate it:

**Gaussian Process (GP):** a non-parametric model that fits all observed
prices simultaneously and returns a posterior *distribution* over the surface
— a mean estimate μ_GP and an uncertainty σ_GP at every point. σ_GP is high
where data is sparse (illiquid strikes), low where data is dense.

**SVI:** the parametric baseline that market makers use to interpolate the
smile within each maturity slice. Where both models agree that a point is
mispriced, the signal is more credible than when only one flags it.


In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

from gpvol.surface.gp_model import GPSurface
from gpvol.surface.svi_model import SVIParams, calibrate_svi, svi_vol
from gpvol.viz.plots import plot_iv_surface_3d, plot_gp_vs_svi

surface = pd.read_parquet("../data/surface_clean.parquet")
print(f"Loaded surface: {len(surface)} contracts")


Loaded surface: 0 contracts


## 1 · Fit the Gaussian Process

In [2]:
gp = GPSurface(n_restarts_optimizer=3, random_state=0)
gp.fit(surface)
print("GP fitted.")
print(f"  Training points: {len(surface)}")
print(f"  k range: [{surface['log_moneyness'].min():.3f}, {surface['log_moneyness'].max():.3f}]")
print(f"  T range: [{surface['T'].min():.3f}, {surface['T'].max():.3f}] years")


ValueError: Found array with 0 sample(s) (shape=(0, 2)) while a minimum of 1 is required by GaussianProcessRegressor.

## 2 · Predict on a regular grid and visualise

`predict_grid()` returns four 2-D arrays shaped `(n_k, n_T)`:
`K_grid, T_grid, mu_grid, sigma_grid`. These feed directly into `plot_iv_surface_3d`.


In [ ]:
K_grid, T_grid, mu_grid, sigma_grid = gp.predict_grid(n_k=60, n_T=25)

# Market observation scatter (calls only for readability)
calls = surface[surface['option_type'] == 'call']
mu_mkt, sg_mkt = gp.predict(calls['log_moneyness'].values, calls['T'].values)

fig = plot_iv_surface_3d(
    K_grid, T_grid, mu_grid, sigma_grid,
    market_k=calls['log_moneyness'].values,
    market_T=calls['T'].values,
    market_iv=calls['iv'].values,
)
fig.show()


## 3 · Calibrate SVI per maturity slice

In [ ]:
import pandas as pd

svi_params = {}
for expiry, group in surface.groupby('expiry'):
    T = float(group['T'].iloc[0])
    if len(group) < 5:
        continue
    params = calibrate_svi(
        k_vals=group['log_moneyness'].values,
        iv_vals=group['iv'].values,
        T=T,
        n_restarts=15,
        random_state=0,
    )
    svi_params[expiry] = (T, params)
    print(f"  {str(expiry)[:10]}  T={T:.3f}  "
          f"a={params.a:.4f} b={params.b:.4f} rho={params.rho:.3f} "
          f"m={params.m:.3f} sigma={params.sigma:.3f}")


## 4 · Compare GP vs SVI per slice

In [ ]:
# Pick the 3-month expiry (closest to T=0.25)
target_T = 0.25
best_expiry = min(svi_params.keys(),
                  key=lambda e: abs(svi_params[e][0] - target_T))
T_slice, params_slice = svi_params[best_expiry]

k_grid = np.linspace(-0.25, 0.25, 80)
mu_gp_slice, sigma_gp_slice = gp.predict(k_grid, np.full_like(k_grid, T_slice))
mu_svi_slice = svi_vol(k_grid, T_slice, params_slice)

slice_mkt = surface[surface['expiry'] == best_expiry]

fig = plot_gp_vs_svi(
    k_vals=k_grid,
    mu_gp=mu_gp_slice,
    sigma_gp=sigma_gp_slice,
    mu_svi=mu_svi_slice,
    market_k=slice_mkt['log_moneyness'].values,
    market_iv=slice_mkt['iv'].values,
    T_label=f"T ≈ {T_slice:.2f}y",
    title=f"GP vs SVI — {str(best_expiry)[:10]}",
)
fig.show()
